# ✈️ FLIGHT FEATURE ENGINEERING (CELL-BY-CELL)

In [3]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")  # points to mlops/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)


Project root added to sys.path: d:\labmentix\major\TravelGuru\travelguru-v5\backend\app\mlops


In [4]:
from loggings import setup_logging
from exception import dump_exception
from utils.db_utils import read_sql_to_df
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

setup_logging()

flights_df = read_sql_to_df("""
SELECT origin, destination, airline, dep_time, arr_time,
       duration_minutes, price, travel_date, cabin_class
FROM database_ml.flights;
""")

hotels_df = read_sql_to_df("""
SELECT hotel_name, city, rating, price_per_night,
       amenities, star_category
FROM database_ml.hotels;
""")

cabs_df = read_sql_to_df("""
SELECT ride_date, ride_time, pickup_location, drop_location,
       vehicle_type, distance_km, price,
       driver_rating, customer_rating
FROM database_ml.cabs;
""")

2026-01-17 00:28:46,975 utils.db_utils INFO Running query: 
SELECT origin, destination, airline, dep_time, arr_time,
       duration_minute


d:\labmentix\major\TravelGuru\travelguru-v5\backend\app\mlops\utils\db_utils.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn, params=params)


2026-01-17 00:28:50,535 utils.db_utils INFO Loaded 2906 rows
2026-01-17 00:28:50,537 utils.db_utils INFO Running query: 
SELECT hotel_name, city, rating, price_per_night,
       amenities, star_catego
2026-01-17 00:28:52,852 utils.db_utils INFO Loaded 1101 rows
2026-01-17 00:28:52,857 utils.db_utils INFO Running query: 
SELECT ride_date, ride_time, pickup_location, drop_location,
       vehicle_typ
2026-01-17 00:29:03,129 utils.db_utils INFO Loaded 33484 rows


Select Required Columns (Lock Schema)

In [5]:
flight_df = flights_df.copy()

flight_df = flight_df[
    [
        "origin",
        "destination",
        "airline",
        "dep_time",
        "arr_time",
        "duration_minutes",
        "price",
        "travel_date",
    ]
]

flight_df.head()


,origin,destination,airline,dep_time,arr_time,duration_minutes,price,travel_date
0,BOM,DEL,IndiGo,1900-01-01 08:30:00,1900-01-01 10:25:00,115,6153.0,2022-02-14
1,BOM,DEL,IndiGo,1900-01-01 07:45:00,1900-01-01 09:50:00,125,5943.0,2022-02-14
2,BOM,DEL,Vistara,1900-01-01 12:25:00,1900-01-01 14:30:00,125,6249.0,2022-02-14
3,BOM,DEL,IndiGo,1900-01-01 10:05:00,1900-01-01 12:15:00,130,5943.0,2022-02-14
4,BOM,DEL,IndiGo,1900-01-01 13:40:00,1900-01-01 15:50:00,130,5943.0,2022-02-14


Datetime Feature Extraction

In [6]:
flight_df["dep_hour"] = flight_df["dep_time"].dt.hour
flight_df["arr_hour"] = flight_df["arr_time"].dt.hour

flight_df["travel_date"] = pd.to_datetime(flight_df["travel_date"])
flight_df["travel_day"] = flight_df["travel_date"].dt.dayofweek  # 0=Mon

flight_df.drop(columns=["dep_time", "arr_time", "travel_date"], inplace=True)

flight_df.head()


,origin,destination,airline,duration_minutes,price,dep_hour,arr_hour,travel_day
0,BOM,DEL,IndiGo,115,6153.0,8,10,0
1,BOM,DEL,IndiGo,125,5943.0,7,9,0
2,BOM,DEL,Vistara,125,6249.0,12,14,0
3,BOM,DEL,IndiGo,130,5943.0,10,12,0
4,BOM,DEL,IndiGo,130,5943.0,13,15,0


Separate Feature Types

In [7]:
flight_numeric_features = [
    "duration_minutes",
    "price",
    "dep_hour",
    "arr_hour",
    "travel_day",
]

flight_categorical_features = [
    "origin",
    "destination",
    "airline",
]

flight_df[flight_numeric_features + flight_categorical_features].head()


,duration_minutes,price,dep_hour,arr_hour,travel_day,origin,destination,airline
0,115,6153.0,8,10,0,BOM,DEL,IndiGo
1,125,5943.0,7,9,0,BOM,DEL,IndiGo
2,125,6249.0,12,14,0,BOM,DEL,Vistara
3,130,5943.0,10,12,0,BOM,DEL,IndiGo
4,130,5943.0,13,15,0,BOM,DEL,IndiGo


Feature Preprocessing Pipeline

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [9]:
flight_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), flight_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), flight_categorical_features),
    ]
)


Transform Features

In [10]:
flight_features = flight_preprocessor.fit_transform(flight_df)

flight_features.shape


(2906, 23)

Save Preprocessor (MLOps Artifact)

In [11]:
import joblib
import os

artifact_dir = "../components/training/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(
    flight_preprocessor,
    os.path.join(artifact_dir, "flight_preprocessor.pkl")
)


['../components/training/artifacts\\flight_preprocessor.pkl']

Define Scoring Logic (CORE ML)

In [12]:
def compute_flight_score(row, user_pref):
    score = 0.0

    # Lower price preferred
    score += -0.4 * row["price"]

    # Shorter duration preferred
    score += -0.3 * row["duration_minutes"]

    # Airline preference
    if row["airline"] in user_pref.get("preferred_airlines", []):
        score += 0.2

    # Departure time preference
    preferred_hours = user_pref.get("preferred_dep_hours", [])
    if row["dep_hour"] in preferred_hours:
        score += 0.1

    return score


Generate Flight Recommendations (TOP-K

In [13]:
user_flight_preferences = {
    "preferred_airlines": ["IndiGo", "Vistara"],
    "preferred_dep_hours": [6, 7, 8, 9],
}


In [14]:
flight_df["score"] = flight_df.apply(
    lambda x: compute_flight_score(x, user_flight_preferences),
    axis=1,
)

top_flights = flight_df.sort_values("score", ascending=False).head(2)

top_flights


,origin,destination,airline,duration_minutes,price,dep_hour,arr_hour,travel_day,score
2859,MAA,BOM,Air India,105,1830.0,6,8,6,-763.4
2772,MAA,BOM,Air India,105,1830.0,6,8,3,-763.4


#🏨 HOTEL FEATURE ENGINEERING (CELL-BY-CELL)

Select & Clean Columns (LOCKED)

In [15]:
hotel_df = hotels_df.copy()

# Drop unusable columns
hotel_df.drop(columns=["hotel_name", "amenities"], inplace=True)

hotel_df = hotel_df[
    [
        "city",
        "rating",
        "price_per_night",
        "star_category",
    ]
]

hotel_df.head()


,city,rating,price_per_night,star_category
0,kochi,4.6,8854.0,5
1,kochi,4.5,6441.0,4
2,kochi,3.8,831.0,4
3,kochi,4.2,2768.0,4
4,kochi,4.5,8938.0,4


Sanity Check

In [16]:
hotel_df.isnull().sum(), hotel_df.dtypes


(city               0
 rating             0
 price_per_night    0
 star_category      0
 dtype: int64,
 city                object
 rating             float64
 price_per_night    float64
 star_category        int64
 dtype: object)

Feature Type Separation

In [18]:
hotel_numeric_features = [
    "rating",
    "price_per_night",
]

hotel_ordinal_features = [
    "star_category",  # ordered
]

hotel_categorical_features = [
    "city",
]

hotel_df.head()


,city,rating,price_per_night,star_category
0,kochi,4.6,8854.0,5
1,kochi,4.5,6441.0,4
2,kochi,3.8,831.0,4
3,kochi,4.2,2768.0,4
4,kochi,4.5,8938.0,4


Preprocessing Pipeline

In [19]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


In [20]:
hotel_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), hotel_numeric_features),
        ("ord", StandardScaler(), hotel_ordinal_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), hotel_categorical_features),
    ]
)


Transform Features

In [21]:
hotel_features = hotel_preprocessor.fit_transform(hotel_df)

hotel_features.shape


(1101, 54)

Save Preprocessor (MLOps Artifact)

In [22]:
import joblib
import os

artifact_dir = "../components/training/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(
    hotel_preprocessor,
    os.path.join(artifact_dir, "hotel_preprocessor.pkl")
)


['../components/training/artifacts\\hotel_preprocessor.pkl']

Hotel Scoring Logic (CORE)

In [23]:
def compute_hotel_score(row, user_pref):
    score = 0.0

    # Lower price preferred
    score += -0.4 * row["price_per_night"]

    # Higher rating preferred
    score += 0.4 * row["rating"]

    # Star category preference
    preferred_stars = user_pref.get("preferred_star_category")
    if preferred_stars is not None:
        score += -0.1 * abs(row["star_category"] - preferred_stars)

    # City match
    if row["city"] == user_pref.get("preferred_city"):
        score += 0.3

    return score


Generate Hotel Recommendations (TOP-K)

In [24]:
user_hotel_preferences = {
    "preferred_city": "Goa",
    "preferred_star_category": 4,
}


In [25]:
hotel_df["score"] = hotel_df.apply(
    lambda x: compute_hotel_score(x, user_hotel_preferences),
    axis=1,
)

top_hotels = hotel_df.sort_values("score", ascending=False).head(2)

top_hotels


,city,rating,price_per_night,star_category,score
575,jamshedpur,4.5,448.0,4,-177.40
445,chandigarh,3.4,461.0,3,-183.14


In [26]:
cab_df = cabs_df.copy()

cab_df = cab_df[
    [
        "ride_date",
        "ride_time",
        "pickup_location",
        "drop_location",
        "vehicle_type",
        "distance_km",
        "price",
        "driver_rating",
        "customer_rating",
    ]
]

cab_df.head()


,ride_date,ride_time,pickup_location,drop_location,vehicle_type,distance_km,price,driver_rating,customer_rating
0,2024-01-28,06:00:00,Area-3,Area-2,Auto,28.50,868.06,4.4,4.4
1,2024-01-17,21:00:00,Area-38,Area-26,Prime Sedan,25.18,348.04,4.5,4.7
2,2024-01-30,00:00:00,Area-47,Area-8,eBike,17.67,56.33,3.6,3.3
3,2024-01-22,08:00:00,Area-11,Area-35,Prime SUV,24.94,1971.38,3.2,3.7
4,2024-01-12,07:00:00,Area-43,Area-42,Prime Sedan,13.15,1130.15,3.1,3.0


In [28]:
# ride_date is string → convert
cab_df["ride_date"] = pd.to_datetime(cab_df["ride_date"])
cab_df["ride_weekday"] = cab_df["ride_date"].dt.dayofweek  # 0 = Monday

# ride_time is already datetime.time → extract hour directly
cab_df["ride_hour"] = cab_df["ride_time"].apply(lambda x: x.hour)

# drop original columns
cab_df.drop(columns=["ride_date", "ride_time"], inplace=True)

cab_df.head()


,pickup_location,drop_location,vehicle_type,distance_km,price,driver_rating,customer_rating,ride_weekday,ride_hour
0,Area-3,Area-2,Auto,28.50,868.06,4.4,4.4,6,6
1,Area-38,Area-26,Prime Sedan,25.18,348.04,4.5,4.7,2,21
2,Area-47,Area-8,eBike,17.67,56.33,3.6,3.3,1,0
3,Area-11,Area-35,Prime SUV,24.94,1971.38,3.2,3.7,0,8
4,Area-43,Area-42,Prime Sedan,13.15,1130.15,3.1,3.0,4,7


In [29]:
cab_numeric_features = [
    "distance_km",
    "price",
    "driver_rating",
    "customer_rating",
    "ride_hour",
    "ride_weekday",
]

cab_categorical_features = [
    "pickup_location",
    "drop_location",
    "vehicle_type",
]

cab_df.head()


,pickup_location,drop_location,vehicle_type,distance_km,price,driver_rating,customer_rating,ride_weekday,ride_hour
0,Area-3,Area-2,Auto,28.50,868.06,4.4,4.4,6,6
1,Area-38,Area-26,Prime Sedan,25.18,348.04,4.5,4.7,2,21
2,Area-47,Area-8,eBike,17.67,56.33,3.6,3.3,1,0
3,Area-11,Area-35,Prime SUV,24.94,1971.38,3.2,3.7,0,8
4,Area-43,Area-42,Prime Sedan,13.15,1130.15,3.1,3.0,4,7


In [30]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


In [31]:
cab_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), cab_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cab_categorical_features),
    ]
)


In [33]:
cab_features = cab_preprocessor.fit_transform(cab_df)

cab_features.shape


(33484, 113)

In [34]:
import joblib
import os

artifact_dir = "../components/training/artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(
    cab_preprocessor,
    os.path.join(artifact_dir, "cab_preprocessor.pkl")
)


['../components/training/artifacts\\cab_preprocessor.pkl']

In [35]:
def compute_cab_score(row, user_pref):
    score = 0.0

    # Lower price preferred
    score += -0.3 * row["price"]

    # Shorter distance preferred
    score += -0.2 * row["distance_km"]

    # Better driver rating
    score += 0.3 * row["driver_rating"]

    # Vehicle preference
    if row["vehicle_type"] in user_pref.get("preferred_vehicle_types", []):
        score += 0.2

    # Pickup-drop match
    if (
        row["pickup_location"] == user_pref.get("pickup_location")
        and row["drop_location"] == user_pref.get("drop_location")
    ):
        score += 0.2

    return score


In [36]:
user_cab_preferences = {
    "preferred_vehicle_types": ["Sedan", "SUV"],
    "pickup_location": "Airport",
    "drop_location": "Hotel Area",
}


In [37]:
cab_df["score"] = cab_df.apply(
    lambda x: compute_cab_score(x, user_cab_preferences),
    axis=1,
)

top_cabs = cab_df.sort_values("score", ascending=False).head(2)

top_cabs


,pickup_location,drop_location,vehicle_type,distance_km,price,driver_rating,customer_rating,ride_weekday,ride_hour,score
27702,Area-48,Area-38,eBike,2.72,51.77,4.5,3.2,6,2,-14.725
23674,Area-46,Area-3,Auto,5.34,51.41,4.7,4.6,3,4,-15.081
